# M5.2 · GAN

İki ağın oyunu. 1-boyutlu bir GAN'da kayıpları ve gradyanları elle hesaplıyor, saf Python'da "Dirac GAN" ile eşzamanlı gradient descent'in neden döndüğünü ve R1 cezasının nasıl yakınsattığını görüyor, NumPy'da optimal discriminator ile JSD bağını ve doymuş/doymamış gradyan farkını sayısal olarak doğruluyor, PyTorch'ta 8 Gauss'lu halka üzerinde bir GAN eğitiyor ve en sonda WGAN-GP gradient penalty'sini örnek başı döngü olmadan hesaplıyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m5-2-gan.html](https://m-gonul.github.io/pytorch-lab/m5-2-gan.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m5_2_gan.py
CHECKS = {
 "b": "import math\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy/torch kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n\n    def gr():\n        f = lambda th, ps: -math.log(1 + math.exp(-ps * th))\n        h = 1e-6\n        for th, ps in ((1.0, 1.0), (0.3, -2.0), (-1.5, 0.7)):\n            g = need(ns, \"grads\")(th, ps)\n            close(g, [(f(th + h, ps) - f(th - h, ps)) / (2 * h), (f(th, ps + h) - f(th, ps - h)) / (2 * h)], atol=1e-6, what=f\"({th}, {ps})\")\n    R.test(\"grads = sayısal türev\", gr)\n\n    def one():\n        th, ps = need(ns, \"simulate\")(1.0, 1.0, 0.1, 1)\n        s = 1 / (1 + math.exp(1.0))\n        close([th, ps], [1 - 0.1 * s, 1 + 0.1 * s])\n        th, ps = ns[\"simulate\"](1.0, 1.0, 0.1, 1, alternating=True)\n        th1 = 1 - 0.1 * s\n        close([th, ps], [th1, 1 + 0.1 * th1 / (1 + math.exp(th1))], what=\"alternatif adım\")\n    R.test(\"tek adım (eşzamanlı ve alternatif)\", one)\n\n    def dyn():\n        r = lambda t: math.hypot(*t)\n        sim = r(need(ns, \"simulate\")(1.0, 1.0, 0.1, 3000))\n        alt = r(ns[\"simulate\"](1.0, 1.0, 0.1, 3000, alternating=True))\n        reg = r(ns[\"simulate\"](1.0, 1.0, 0.1, 3000, gamma=1.0))\n        ensure(sim > 2.5, f\"eşzamanlı GD uzaklaşmalı: yarıçap {sim:.3f}\")\n        ensure(0.5 < alt < 2.5, f\"alternatif sınırlı kalmalı: {alt:.3f}\")\n        ensure(reg < 1e-3, f\"R1 ile yakınsamalı: {reg:.2e}\")\n        return f\"başlangıç yarıçapı 1.414 → eşzamanlı {sim:.3f}, alternatif {alt:.3f}, R1 {reg:.1e}\"\n    R.test(\"üç dinamik\", dyn)\n",
 "c": "import numpy as np\nfrom labcheck import close, no_loops, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n    R.test(\"torch kullanılmıyor\", lambda: no_imports(src, {\"torch\"}))\n    rng = np.random.default_rng(0)\n    p = rng.random(30); p[[3, 7]] = 0; p /= p.sum()\n    q = rng.random(30); q[[7, 12]] = 0; q /= q.sum()\n\n    def js():\n        m = 0.5 * (p + q)\n        ref = 0.5 * sum(a * np.log(a / b) for a, b in zip(p, m) if a > 0) + 0.5 * sum(a * np.log(a / b) for a, b in zip(q, m) if a > 0)\n        close(need(ns, \"jsd\")(p, q), ref)\n        close(ns[\"jsd\"](p, p), 0.0, what=\"JSD(p, p)\")\n        close(ns[\"jsd\"](np.array([1.0, 0]), np.array([0, 1.0])), np.log(2), what=\"ayrık destekler → log 2\")\n    R.test(\"jsd (sıfır kutular dahil)\", js)\n\n    def eqn():\n        D = need(ns, \"optimal_d\")(p, q)\n        ensure(np.all(np.isfinite(D)) and D[7] == 0.5, \"p = q = 0 kutusunda 0.5\")\n        Dc = np.clip(D, 1e-12, 1 - 1e-12)\n        close(need(ns, \"value_expect\")(p, q, Dc), -np.log(4) + 2 * ns[\"jsd\"](p, q), atol=1e-8)\n    R.test(\"V(D*) = −log 4 + 2·JSD\", eqn)\n\n    def maxim():\n        D = np.clip(need(ns, \"optimal_d\")(p, q), 1e-9, 1 - 1e-9)\n        v = need(ns, \"value_expect\")(p, q, D)\n        for _ in range(20):\n            Dp = np.clip(D + rng.normal(0, 0.05, D.shape), 1e-6, 1 - 1e-6)\n            ensure(ns[\"value_expect\"](p, q, Dp) <= v + 1e-12, \"bozulmuş D daha yüksek değer verdi\")\n    R.test(\"D* gerçekten maksimum\", maxim)\n\n    def grads():\n        sat, ns_ = need(ns, \"gen_logit_grads\")(np.array([-5.0, 0.0, 5.0]))\n        close(sat, [-0.006693, -0.5, -0.993307], atol=1e-6, what=\"doymuş\")\n        close(ns_, [-0.993307, -0.5, -0.006693], atol=1e-6, what=\"doymamış\")\n        return f\"logit −5'te (D emin: sahte) doymuş {sat[0]:.4f}, doymamış {ns_[0]:.4f}: ~150 kat\"\n    R.test(\"doymuş vs doymamış gradyan\", grads)\n",
 "d": "import math\nimport torch\nimport torch.nn.functional as F\nfrom labcheck import close, need, ensure\n\n\ndef ring(n):\n    k = torch.randint(0, 8, (n,))\n    ang = k * 2 * math.pi / 8\n    return torch.stack([2 * ang.cos(), 2 * ang.sin()], 1) + 0.05 * torch.randn(n, 2)\n\n\ndef check(ns, R):\n    def losses():\n        torch.manual_seed(0)\n        G, D = need(ns, \"Generator\", \"sınıf\")(), need(ns, \"Discriminator\", \"sınıf\")()\n        real = ring(16); fake = G(torch.randn(16, 2))\n        lr, lf = D(real), D(fake)\n        ensure(tuple(lr.shape) in ((16, 1), (16,)), \"ayırıcı (B, 1) logit döndürmeli\")\n        want_d = F.softplus(-lr).mean() + F.softplus(lf).mean()\n        close(need(ns, \"d_loss\")(D, real, fake), want_d, rtol=1e-5)\n        close(need(ns, \"g_loss\")(D, fake), F.softplus(-lf).mean(), rtol=1e-5)\n    R.test(\"kayıp formülleri\", losses)\n\n    def detach():\n        torch.manual_seed(0)\n        G, D = ns[\"Generator\"](), ns[\"Discriminator\"]()\n        need(ns, \"d_loss\")(D, ring(8), G(torch.randn(8, 2))).backward()\n        ensure(all(p.grad is None or p.grad.abs().sum() == 0 for p in G.parameters()), \"d_loss G'ye gradyan taşımamalı (fake.detach())\")\n        ensure(any(p.grad is not None for p in D.parameters()), \"d_loss D'ye gradyan vermeli\")\n    R.test(\"d_loss sahte örnekleri detach ediyor\", detach)\n\n    def run(seed):\n        torch.manual_seed(seed)\n        G, D = need(ns, \"Generator\", \"sınıf\")(), need(ns, \"Discriminator\", \"sınıf\")()\n        need(ns, \"train_gan\")(G, D, ring, steps=3000)\n        with torch.no_grad():\n            s = G(torch.randn(2000, 2))\n        ang = torch.arange(8) * 2 * math.pi / 8\n        C = torch.stack([2 * ang.cos(), 2 * ang.sin()], 1)\n        dmin, k = torch.cdist(s, C).min(1)\n        good = dmin < 0.2\n        return (torch.bincount(k[good], minlength=8) >= 40).sum().item(), good.float().mean().item()\n\n    def train():\n        tried = []\n        for seed in (1, 2):\n            modes, q = run(seed)\n            tried.append(f\"tohum {seed}: {modes}/8 mod, %{100 * q:.0f}\")\n            if modes >= 7 and q > 0.6:\n                return f\"{modes}/8 mod, bir moda 0.2'den yakın örnek oranı %{100 * q:.0f}\"\n        raise AssertionError(\"mod çöküşü ya da dağınık örnekler — \" + \"; \".join(tried))\n    R.test(\"8 Gauss halkası: modlar kapsanıyor\", train)\n",
 "e": "import torch\nfrom torch import nn\nfrom labcheck import close, max_loops, need, ensure, bench, fmt_time\n\n\ndef gp_loop(D, real, fake, alpha):\n    vals = []\n    for i in range(real.shape[0]):\n        x = (alpha[i] * real[i] + (1 - alpha[i]) * fake[i]).detach().requires_grad_(True)\n        g, = torch.autograd.grad(D(x[None]).sum(), x, create_graph=True)\n        vals.append((g.norm() - 1) ** 2)\n    return torch.stack(vals).mean()\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"gradient_penalty döngüsüz\", lambda: max_loops(src, 0, func=\"gradient_penalty\"))\n    torch.manual_seed(0)\n    D = nn.Sequential(nn.Linear(4, 32), nn.Tanh(), nn.Linear(32, 32), nn.Tanh(), nn.Linear(32, 1)).double()\n    g = torch.Generator().manual_seed(0)\n    real, fake = torch.randn(16, 4, generator=g, dtype=torch.float64), torch.randn(16, 4, generator=g, dtype=torch.float64)\n    alpha = torch.rand(16, 1, generator=g, dtype=torch.float64)\n    R.test(\"örnek başı döngüyle aynı\", lambda: close(need(ns, \"gradient_penalty\")(D, real, fake, alpha), gp_loop(D, real, fake, alpha)))\n\n    def lin():\n        L = nn.Linear(4, 1, bias=False).double()\n        with torch.no_grad():\n            L.weight.copy_(torch.tensor([[3.0, 0, 4, 0]]))\n        close(need(ns, \"gradient_penalty\")(L, real, fake, alpha), 16.0, what=\"‖w‖ = 5 → (5 − 1)² = 16\")\n    R.test(\"doğrusal critic: (‖w‖ − 1)²\", lin)\n\n    def to_params():\n        D.zero_grad()\n        need(ns, \"gradient_penalty\")(D, real, fake, alpha).backward()\n        ensure(all(p.grad is not None and p.grad.abs().sum() > 0 for p in D.parameters() if p.dim() > 1), \"ceza D'nin ağırlıklarına gradyan vermeli (create_graph=True)\")\n        z = lambda p: torch.zeros_like(p) if p.grad is None else p.grad.clone()\n        a = [z(p) for p in D.parameters()]\n        D.zero_grad(set_to_none=True); gp_loop(D, real, fake, alpha).backward()\n        for x, y in zip(a, [z(p) for p in D.parameters()]):\n            close(x, y, what=\"parametre gradyanı\")\n    R.test(\"ikinci türev: D parametrelerine gradyan\", to_params)\n\n    def bn():\n        Dbn = nn.Sequential(nn.Linear(4, 16), nn.BatchNorm1d(16), nn.Tanh(), nn.Linear(16, 1)).double().train()\n        a = need(ns, \"gradient_penalty\")(Dbn, real, fake, alpha).item()\n        Dbn.eval(); b = ns[\"gradient_penalty\"](Dbn, real, fake, alpha).item()\n        return f\"BatchNorm'lu critic: train modunda {a:.4f}, eval modunda {b:.4f}. Örnekler birbirine bağlanınca toplamın gradyanı örnek başı gradyan olmaz.\"\n    R.test(\"not: BatchNorm örnekleri birbirine bağlar\", bn)\n    if \"gradient_penalty\" not in ns:\n        return\n    D32 = nn.Sequential(nn.Linear(64, 256), nn.Tanh(), nn.Linear(256, 1))\n    r, f, a = torch.randn(256, 64), torch.randn(256, 64), torch.rand(256, 1)\n    t_loop = bench(lambda: gp_loop(D32, r, f, a), repeat=3, number=1)\n    t_vec = bench(lambda: ns[\"gradient_penalty\"](D32, r, f, a), repeat=3)\n    R.table(\"B = 256, d = 64\", [\"yöntem\", \"süre\"], [[\"örnek başı autograd.grad\", fmt_time(t_loop)], [\"senin (tek çağrı)\", fmt_time(t_vec)]])\n",
 "x1": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngüsüz\", lambda: max_loops(src, 0, func=\"minibatch_features\"))\n\n    def t():\n        rng = np.random.default_rng(0)\n        F = rng.normal(size=(4, 3))\n        T = rng.normal(size=(3, 2, 5))\n        o = need(ns, \"minibatch_features\")(F, T)\n        M = np.einsum(\"na,abc->nbc\", F, T)\n        ref = np.array([[sum(np.exp(-np.abs(M[i, b] - M[j, b]).sum()) for j in range(4)) for b in range(2)] for i in range(4)])\n        close(o, ref, what=\"tanım\")\n        T2 = rng.normal(size=(8, 16, 4))\n        div = rng.normal(size=(64, 8))\n        col = np.repeat(rng.normal(size=(1, 8)), 64, 0) + 0.01 * rng.normal(size=(64, 8))\n        od, oc = ns[\"minibatch_features\"](div, T2), ns[\"minibatch_features\"](col, T2)\n        ensure(oc.mean() > 20 * od.mean(), \"çökmüş batch'te benzerlik öznitelikleri devasa\")\n        R_.note(f\"Çeşitli batch'te o(x) ≈ {od.mean():.2f} (yalnız kendisi ≈ 1 katkı yapar), neredeyse aynı örneklerden oluşan batch'te ≈ {oc.mean():.1f} (≈ batch boyutu 64). Ayırıcı bu özniteliği gerçek batch'lerde düşük gördüğü için çökmüş batch'i tek bakışta reddeder; üretici çeşitlilik üretmeye zorlanır. ProGAN'ın mini-batch std katmanı (M5.7 x3) aynı fikrin parametresiz sürümüdür.\")\n    R_.test(\"mini-batch ayrımı\", t)\n",
 "x2": "import numpy as np\nfrom labcheck import need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        rng = np.random.default_rng(0)\n        B = need(ns, \"ReplayBuffer\")(5)\n        B.add(np.arange(3.0)[:, None])\n        ensure(len(B) == 3, \"uzunluk\")\n        B.add(np.arange(3.0, 7.0)[:, None])\n        ensure(len(B) == 5 and sorted(B.sample(5, rng)[:, 0].tolist()) == [2.0, 3.0, 4.0, 5.0, 6.0], \"dairesel: en eskiler silinir\")\n        B2 = ns[\"ReplayBuffer\"](100)\n        first = need(ns, \"mixed_fakes\")(np.full((10, 1), 0.0), B2, 0.5, rng)\n        ensure(np.all(first == 0) and len(B2) == 10, \"ilk adımda tampon boş → yalnız güncel\")\n        for step in range(1, 30):\n            out = ns[\"mixed_fakes\"](np.full((10, 1), float(step)), B2, 0.5, rng)\n        ensure((out == 29).sum() == 5 and len(B2) == 100, \"yarı güncel, yarı geçmiş\")\n        ages = 29 - out[out != 29]\n        R_.note(f\"Son adımda ayırıcının sahte batch'i: 5 güncel örnek + tampondan 5 örnek (yaşları {sorted(ages.astype(int).tolist())} adım). Tampon kapasitesi ayırıcının 'hafızasını' belirler; çok büyük tampon, üreticinin artık üretmediği örneklere karşı ayrım öğretir (gecikme), çok küçük tampon etkisizdir.\")\n    R_.test(\"deneyim tekrarı\", t)\n"
}

In [ ]:
import importlib, labcheck, checks_m5_2_gan as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

Üreteç $G(z)$ gürültüden örnek üretir, ayırıcı $D(x)\in(0,1)$ gerçek mi sahte mi tahmin eder:

$$
\min_G\max_D\ V(D,G)=\mathbb E_{x\sim p_\text{data}}[\log D(x)]+\mathbb E_{z}[\log(1-D(G(z)))].
$$

Sabit $G$ için en iyi ayırıcı $D^*(x)=\frac{p_\text{data}(x)}{p_\text{data}(x)+p_g(x)}$ ve bu noktada $V(D^*,G)=-\log4+2\,\mathrm{JSD}(p_\text{data}\,\|\,p_g)$: $G$, Jensen–Shannon ıraksamasını küçültüyor.

Pratikte üreteç $\log(1-D(G(z)))$'yi küçültmek yerine $-\log D(G(z))$'yi küçültür (non-saturating loss).

GAN eğitimi bir **oyun**: iki oyuncu aynı anda gradyan adımı atar. Bu, tek bir kaybı küçültmekten farklı dinamikler doğurur; en basit örnek **Dirac GAN**: veri $\delta_0$, üreteç $\delta_\theta$, ayırıcı $D(x)=\sigma(\psi x)$. Değer $V(\theta,\psi)=f(\psi\theta)$, $f(t)=-\log(1+e^{-t})$.

WGAN-GP ayırıcıyı ("critic") 1-Lipschitz'e yaklaştırmak için gradient penalty ekler: $\lambda\,\mathbb E_{\hat x}\big[(\lVert\nabla_{\hat x}D(\hat x)\rVert-1)^2\big]$, $\hat x=\alpha x+(1-\alpha)G(z)$.

## 1 · Elle

<!-- çözüm -->

$D(G(z))=0.5$ noktasında doymuş ve doymamış kayıpların gradyanları aynı çıkıyor. Fark, $D(G(z))$ sıfıra yaklaşınca açılıyor: biri $-D$ ile, diğeri $-(1-D)$ ile orantılı. c aşamasında bunu logit $-5$'te ölçüyorsun.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**$D(x)=\sigma(wx+b)$, $w=1$, $b=0$; $G(z)=az+c$, $a=2$, $c=-1$; gerçek $x=2$, gürültü $z=0.5$**

- `dreal`: $D(x)$
- `ld`: $L_D=-\log D(x)-\log(1-D(G(z)))$
- `dw`: $\partial L_D/\partial w$

**Üreteç, non-saturating kayıp $L_G=-\log D(G(z))$**

- `lg`: $L_G$
- `da`: $\partial L_G/\partial a$
- `dsat`: Doymuş kayıp $\log(1-D(G))$ ile $\partial/\partial a$

In [ ]:
cevaplar = dict(
    dreal=None,
    ld=None,
    dw=None,
    lg=None,
    da=None,
    dsat=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"dreal":[0.880797,0.0005],"ld":[0.820075,0.0005],"dw":[-0.238406,0.0005],"lg":[0.693147,0.0005],"da":[-0.25,0.0005],"dsat":[-0.25,0.0005]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python, **Dirac GAN**. $f'(t)=\sigma(-t)$, dolayısıyla

$$
\partial_\theta V=\psi\,\sigma(-\psi\theta),\qquad \partial_\psi V=\theta\,\sigma(-\psi\theta).
$$

Üreteç $V$'yi küçültür, ayırıcı büyütür (eksi R1 cezası $\tfrac\gamma2\psi^2$: gerçek veride $\nabla_xD$ büyüklüğü $\psi$ ile orantılı).

- `grads(theta, psi)` → `(gθ, gψ)`.
- `simulate(theta, psi, h, steps, gamma=0.0, alternating=False)` → son `(theta, psi)`:
  - eşzamanlı: ikisi de **eski** değerlerle güncellenir: $\theta\leftarrow\theta-h\,g_\theta$, $\psi\leftarrow\psi+h(g_\psi-\gamma\psi)$.
  - `alternating=True`: önce $\theta$ güncellenir, $g_\psi$ **yeni** $\theta$ ile hesaplanır.

Kontrol üç davranışı sınar: eşzamanlı uzaklaşır, alternatif sınırlı kalır, R1 ile $(0,0)$'a yakınsar.

In [ ]:
%%writefile stage_b.py
import math


def sig(t):
    return 1 / (1 + math.exp(-t))


def grads(theta, psi):
    ...


def simulate(theta, psi, h, steps, gamma=0.0, alternating=False):
    ...


for kw in [{}, {"alternating": True}, {"gamma": 1.0}]:
    th, ps = simulate(1.0, 1.0, 0.1, 3000, **kw)
    print(kw, round(th, 4), round(ps, 4), round(math.hypot(th, ps), 4))

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy, **döngü yok**:

- `value_expect(p, q, D)` — ayrık dağılımlar (aynı kutular): $\sum_i p_i\log D_i+\sum_i q_i\log(1-D_i)$.
- `optimal_d(p, q)` — $p/(p+q)$; $p+q=0$ olan kutularda 0.5 (değer etkisiz).
- `jsd(p, q)` — $\tfrac12\mathrm{KL}(p\|m)+\tfrac12\mathrm{KL}(q\|m)$, $m=\tfrac12(p+q)$; $0\log0=0$.
- `gen_logit_grads(l)` — sahte örneklerin ayırıcı logit'leri $l$ için üretecin kaybının logit'e göre türevleri: `(doymuş, doymamış)`. Doymuş kayıp $\log(1-\sigma(l))$, doymamış $-\log\sigma(l)$.

Kontrol, $V(D^*)=-\log4+2\,\mathrm{JSD}$ eşitliğini ve $D^*$'ın gerçekten maksimum olduğunu (her bozulma değeri düşürür) doğrular.

In [ ]:
%%writefile stage_c.py
import numpy as np


def value_expect(p, q, D):
    ...


def optimal_d(p, q):
    ...


def jsd(p, q):
    ...


def gen_logit_grads(l):
    ...


x = np.linspace(-4, 6, 50)
p = np.exp(-0.5 * x ** 2); p /= p.sum()
q = np.exp(-0.5 * (x - 2) ** 2); q /= q.sum()
print(value_expect(p, q, optimal_d(p, q)), -np.log(4) + 2 * jsd(p, q))
print(gen_logit_grads(np.array([-5.0, 0.0, 5.0])))

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

Hedef dağılım: yarıçapı 2 olan çember üzerinde 8 Gauss (σ = 0.05). Mod çöküşünü görmek için klasik deney.

- `Generator(z_dim=2, hidden=128)` ve `Discriminator(hidden=128)` — `Linear → ReLU → Linear → ReLU → Linear`; ayırıcı **logit** döndürür.
- `d_loss(D, real, fake)` — BCE-with-logits: gerçek → 1, sahte → 0. Sahte örnekler buradan $G$'ye gradyan **taşımamalı** (`detach`).
- `g_loss(D, fake)` — non-saturating: sahte → 1 etiketiyle BCE.
- `train_gan(G, D, sample_real, steps=3000, batch=256, lr=1e-3)` — Adam (β₁ = 0.5); her adımda bir D, bir G güncellemesi.

Kontrol 2000 örnek üretip kaç modun kapsandığını ve örneklerin ne kadarının bir moda 0.2'den yakın olduğunu ölçer.

In [ ]:
%%writefile stage_d.py
import math
import torch
import torch.nn.functional as F
from torch import nn


class Generator(nn.Module):
    def __init__(self, z_dim=2, hidden=128):
        super().__init__()
        ...

    def forward(self, z):
        ...


class Discriminator(nn.Module):
    def __init__(self, hidden=128):
        super().__init__()
        ...

    def forward(self, x):
        ...


def d_loss(D, real, fake):
    ...


def g_loss(D, fake):
    ...


def train_gan(G, D, sample_real, steps=3000, batch=256, lr=1e-3):
    ...


def ring(n):
    k = torch.randint(0, 8, (n,))
    ang = k * 2 * math.pi / 8
    return torch.stack([2 * ang.cos(), 2 * ang.sin()], 1) + 0.05 * torch.randn(n, 2)


torch.manual_seed(0)
G, D = Generator(), Discriminator()
train_gan(G, D, ring, steps=1000)
print(G(torch.randn(5, 2)))

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

`gradient_penalty(D, real, fake, alpha)` — `alpha: (B, 1)`.

1. $\hat x=\alpha\,x+(1-\alpha)\,\tilde x$, `requires_grad_(True)`.
2. `g, = torch.autograd.grad(D(x_hat).sum(), x_hat, create_graph=True)` — örnekler bağımsız olduğundan toplamın gradyanı = örnek başı gradyanlar.
3. $\operatorname{mean}_i(\lVert g_i\rVert_2-1)^2$.

**Döngü yok.** `create_graph=True` olmazsa ceza ayırıcının parametrelerine gradyan vermez; kontrol bunu da sınar.

In [ ]:
%%writefile stage_e.py
import torch


def gradient_penalty(D, real, fake, alpha):
    """alpha: (B, 1). Döngü yok."""
    ...


torch.manual_seed(0)
D = torch.nn.Sequential(torch.nn.Linear(2, 32), torch.nn.Tanh(), torch.nn.Linear(32, 1))
print(gradient_penalty(D, torch.randn(8, 2), torch.randn(8, 2), torch.rand(8, 1)))

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · Mini-batch ayrımı (Salimans ve ark. 2016)

**Mini-batch ayrımı** (Salimans ve ark. 2016; Géron): ayırıcıya, bir örneğin batch'teki diğer örneklere ne kadar benzediğini gösteren öznitelikler eklenir; çeşitliliği olmayan (mod çöküşü yaşamış) sahte batch'ler kolayca tanınır. Ara öznitelik $f(x_i)\in\mathbb R^A$, öğrenilen tensör $T\in\mathbb R^{A\times B\times C}$: $M_i=f(x_i)T\in\mathbb R^{B\times C}$, $c_b(x_i,x_j)=\exp(-\|M_{i,b}-M_{j,b}\|_{L_1})$, $o(x_i)_b=\sum_jc_b(x_i,x_j)$ (j = i dahil). Çıktı $[f(x_i), o(x_i)]$.

NumPy, döngüsüz.

- `minibatch_features(F, T)` — `F (N, A)`, `T (A, B, C)` → `(N, B)`.

In [ ]:
%%writefile extra_x1.py
import numpy as np


def minibatch_features(F, T):
    ...

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · Deneyim tekrarı tamponu

**Deneyim tekrarı** (Shrivastava ve ark. 2017; Géron): ayırıcı yalnız son üreticinin örnekleriyle değil, geçmiş üretilmiş örneklerden oluşan bir tampondan da beslenir; son üreticiye aşırı uymasını (ve üretici–ayırıcı arasında modlar arası kovalamacayı) azaltır.

Saf Python + NumPy.

- `class ReplayBuffer(capacity)` — `add(batch)`: satırları sırayla ekle, dolunca en eskilerin üzerine yaz (dairesel); `sample(k, rng)`: içerikten yerine koymadan k satır; `__len__`.
- `mixed_fakes(current, buffer, frac, rng)` — ayırıcının sahte batch'i: `frac` oranı tampondan, kalanı güncel örneklerden; ardından güncel batch tampona eklenir.

In [ ]:
%%writefile extra_x2.py
import numpy as np


class ReplayBuffer:
    def __init__(self, capacity):
        ...

    def add(self, batch):
        ...

    def sample(self, k, rng):
        ...

    def __len__(self):
        ...


def mixed_fakes(current, buffer, frac, rng):
    ...

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

## Makale ödevi: Goodfellow ve ark. (2014): ilk GAN

[Goodfellow ve ark. — Generative Adversarial Nets (NeurIPS 2014)](https://arxiv.org/abs/1406.2661)

Makale Şekil 1'deki 1B örneği tekrarla: veri N(4, 0.5²), üretici gürültüyü doğrusal olmayan bir MLP ile eşler; eğitim boyunca p_g, p_data ve D'yi aynı eksende çiz ve D'nin 1/2'ye düzleştiğini göster. Algoritma 1'deki k (ayırıcı adım sayısı) ∈ {1, 5} ve doymuş/doymamış üretici kaybını karşılaştır; Parzen pencereli log-olabilirlik tahminini (Tablo 1) MNIST'te küçük bir GAN için hesapla ve sınırlılıklarını tartış.


**Adımlar**

1. Parzen bant genişliğini doğrulama kümesinde seç.

**Kabul ölçütleri**

- 1B'de p_g veriye yakınsar, D ≈ 0.5.
- Doymuş kayıpla eğitimin başlangıçta takıldığı gözlemlenir.

## Makale ödevi: Salimans ve ark. (2016): GAN eğitimi için iyileştirilmiş teknikler

[Salimans, Goodfellow, Zaremba, Cheung, Radford, Chen — Improved Techniques for Training GANs (NeurIPS 2016)](https://arxiv.org/abs/1606.03498)

M5.2 d aşamasındaki 8 Gauss halkası GAN'ına makalenin tekniklerini sırayla ekle: öznitelik eşleme (Denk. 4), mini-batch ayrımı (x1), tarihsel ortalama (θ'yı geçmiş ortalamasına yakın tutan ceza), tek taraflı etiket yumuşatma (gerçek etiketi 0.9) ve sanal batch normalizasyonu. Her tekniğin kapsanan mod sayısına ve eğitim kararlılığına etkisini 10 tohum üzerinden raporla.


**Adımlar**

1. Kapsanan mod: her moda en az %1 örnek düşüyorsa sayılır.

**Kabul ölçütleri**

- Mini-batch ayrımı mod sayısını belirgin artırır.
- Tekniklerin birleşimi tohumlar arası varyansı azaltır.

## Kaynaklar

- [Goodfellow ve ark. — Generative Adversarial Nets](https://arxiv.org/abs/1406.2661) — arXiv
- [Mescheder, Geiger, Nowozin — Which Training Methods for GANs do actually Converge? (Dirac GAN, R1)](https://arxiv.org/abs/1801.04406) — arXiv
- [Gulrajani ve ark. — Improved Training of Wasserstein GANs (WGAN-GP)](https://arxiv.org/abs/1704.00028) — arXiv
- [Prince — Understanding Deep Learning, Bölüm 15 (GAN)](https://udlbook.github.io/udlbook/) — kitap ©; notebook'lar MIT
- [Stanford CS236 — Deep Generative Models](https://deepgenerativemodels.github.io/) — doğrulanmadı